In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

data_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics.csv"
)

df = pd.read_csv("C:\\Users\\nitin\\Desktop\\ML Projects\\Customer Churn\\data\\processed\\telco_churn_analytics.csv")

print("Dataset shape:", df.shape)
display(df.head())

In [ ]:
print("Number of customers:", len(df))
print("Number of columns:", df.shape[1])
print("Duplicate customers:", df["customerid"].duplicated().sum())
print("Overall churn rate:", round(df["churn_value"].mean() * 100, 2), "%")

In [ ]:
missing_summary = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame("missing_count")
)

missing_summary["missing_percentage"] = (
    missing_summary["missing_count"] / len(df) * 100
).round(2)

display(
    missing_summary[
        missing_summary["missing_count"] > 0
    ]
)

In [ ]:
numerical_columns = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "cltv"
]

display(
    df[numerical_columns]
    .describe()
    .T
    .round(2)
)

In [ ]:
categorical_columns = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents",
    "internet_service",
    "contract",
    "payment_method",
    "tenure_group",
    "monthly_charge_group"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))

In [ ]:
def churn_summary(data, group_column):
    """
    Calculate customer count, churned customers and churn rate
    for each category of the selected column.
    """
    
    summary = (
        data.groupby(
            group_column,
            observed=False,
            dropna=False
        )
        .agg(
            customer_count=("customerid", "count"),
            churned_customers=("churn_value", "sum"),
            churn_rate=("churn_value", "mean")
        )
        .reset_index()
    )

    summary["churn_rate_percentage"] = (
        summary["churn_rate"] * 100
    ).round(2)

    summary["customer_percentage"] = (
        summary["customer_count"] / len(data) * 100
    ).round(2)

    return summary.sort_values(
        "churn_rate",
        ascending=False
    )

In [ ]:
display(churn_summary(df, "contract"))

In [ ]:
def plot_churn_rate(data, group_column, title=None):
    """
    Plot churn rate by a categorical variable.
    """

    summary = churn_summary(data, group_column)

    plot_data = summary.sort_values(
        "churn_rate_percentage",
        ascending=True
    )

    plt.figure(figsize=(9, 5))

    bars = plt.barh(
        plot_data[group_column].astype(str),
        plot_data["churn_rate_percentage"]
    )

    plt.xlabel("Churn Rate (%)")
    plt.ylabel(group_column.replace("_", " ").title())

    if title is None:
        title = (
            f"Churn Rate by "
            f"{group_column.replace('_', ' ').title()}"
        )

    plt.title(title)

    for bar, value in zip(
        bars,
        plot_data["churn_rate_percentage"]
    ):
        plt.text(
            bar.get_width() + 0.5,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.1f}%",
            va="center"
        )

    plt.tight_layout()
    plt.show()

In [ ]:
churn_distribution = (
    df.groupby("churn_label")
      .agg(
          customer_count=("customerid", "count")
      )
      .reset_index()
)

churn_distribution["percentage"] = (
    churn_distribution["customer_count"]
    / len(df)
    * 100
).round(2)

display(churn_distribution)

In [ ]:
plt.figure(figsize=(7, 5))

bars = plt.bar(
    churn_distribution["churn_label"],
    churn_distribution["customer_count"]
)

plt.title("Overall Customer Churn Distribution")
plt.xlabel("Churn Status")
plt.ylabel("Number of Customers")

for bar, count, percentage in zip(
    bars,
    churn_distribution["customer_count"],
    churn_distribution["percentage"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 50,
        f"{count}\n({percentage:.1f}%)",
        ha="center"
    )

plt.tight_layout()
plt.show()

### Observation

The dataset contains 7,043 customers, of whom 1,869 churned. The
overall churn rate is approximately 26.54%.

This also indicates a moderate class imbalance: non-churned customers
represent approximately 73.46% of the dataset. Therefore, accuracy alone
will not be sufficient for evaluating the churn-prediction model.
Metrics such as recall, precision, F1-score, ROC-AUC and PR-AUC will
also be considered.

In [ ]:
contract_summary = churn_summary(df, "contract")

display(contract_summary)

In [ ]:
plot_churn_rate(
    df,
    "contract",
    title="Customer Churn Rate by Contract Type"
)

In [ ]:
contract_counts = (
    df["contract"]
    .value_counts()
    .sort_values(ascending=True)
)

plt.figure(figsize=(9, 5))

bars = plt.barh(
    contract_counts.index,
    contract_counts.values
)

plt.title("Number of Customers by Contract Type")
plt.xlabel("Number of Customers")
plt.ylabel("Contract Type")

for bar, value in zip(bars, contract_counts.values):
    plt.text(
        bar.get_width() + 20,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center"
    )

plt.tight_layout()
plt.show()

### Observation

Month-to-month customers have the highest churn rate, while customers
with one-year and two-year contracts show substantially lower churn.

This suggests that customers without a long-term commitment are an
important retention segment. However, this is an association rather
than proof that the contract type itself causes churn.

A possible business action is to encourage high-risk month-to-month
customers to move to annual contracts through targeted discounts or
loyalty benefits.

In [ ]:
tenure_order = [
    "0-6 months",
    "7-12 months",
    "13-24 months",
    "25-48 months",
    "49-72 months"
]

df["tenure_group"] = pd.Categorical(
    df["tenure_group"],
    categories=tenure_order,
    ordered=True
)

In [ ]:
tenure_summary = (
    df.groupby(
        "tenure_group",
        observed=False
    )
    .agg(
        customer_count=("customerid", "count"),
        churned_customers=("churn_value", "sum"),
        churn_rate=("churn_value", "mean")
    )
    .reset_index()
)

tenure_summary["churn_rate_percentage"] = (
    tenure_summary["churn_rate"] * 100
).round(2)

display(tenure_summary)

In [ ]:
plt.figure(figsize=(10, 5))

bars = plt.bar(
    tenure_summary["tenure_group"].astype(str),
    tenure_summary["churn_rate_percentage"]
)

plt.title("Customer Churn Rate by Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=25)

for bar, value in zip(
    bars,
    tenure_summary["churn_rate_percentage"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()


In [ ]:
tenure_by_churn = (
    df.groupby("churn_label")
      .agg(
          customer_count=("customerid", "count"),
          average_tenure=("tenure_months", "mean"),
          median_tenure=("tenure_months", "median")
      )
      .reset_index()
)

tenure_by_churn[
    ["average_tenure", "median_tenure"]
] = tenure_by_churn[
    ["average_tenure", "median_tenure"]
].round(2)

display(tenure_by_churn)

In [ ]:
non_churned_tenure = df.loc[
    df["churn_value"] == 0,
    "tenure_months"
]

churned_tenure = df.loc[
    df["churn_value"] == 1,
    "tenure_months"
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    [non_churned_tenure, churned_tenure],
    tick_labels=["Not Churned", "Churned"]
)

plt.title("Tenure Distribution by Churn Status")
plt.xlabel("Customer Status")
plt.ylabel("Tenure in Months")

plt.tight_layout()
plt.show()

### Observation

The churn rate is highest among customers in the earliest tenure group
and generally decreases as tenure increases.

The median and average tenure of churned customers should also be
compared with those of retained customers. This helps determine whether
customer loss is concentrated during the early stages of the customer
relationship.

The result suggests that onboarding and early-life retention campaigns
may be valuable, particularly during the first few months after a
customer joins.

In [ ]:
payment_summary = churn_summary(
    df,
    "payment_method"
)

display(payment_summary)

In [ ]:
plot_churn_rate(
    df,
    "payment_method",
    title="Customer Churn Rate by Payment Method"
)

In [ ]:
payment_churn_count = (
    df[df["churn_value"] == 1]
    ["payment_method"]
    .value_counts()
    .sort_values(ascending=True)
)

plt.figure(figsize=(9, 5))

bars = plt.barh(
    payment_churn_count.index,
    payment_churn_count.values
)

plt.title("Number of Churned Customers by Payment Method")
plt.xlabel("Number of Churned Customers")
plt.ylabel("Payment Method")

for bar, value in zip(
    bars,
    payment_churn_count.values
):
    plt.text(
        bar.get_width() + 10,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center"
    )

plt.tight_layout()
plt.show()

### Observation

The churn rate differs considerably across payment methods. The payment
method with the highest churn rate should be treated as a risk indicator,
but it should not automatically be interpreted as the direct reason for
churn.

Payment method may be associated with other factors such as contract
type, customer tenure or automatic-payment enrolment. These relationships
will be investigated later through multivariate modelling.

In [ ]:
eda_findings = pd.DataFrame(
    columns=[
        "analysis_area",
        "key_finding",
        "possible_business_action",
        "caution"
    ]
)

eda_findings

In [ ]:
new_findings = pd.DataFrame(
    [
        {
            "analysis_area": "Contract",
            "key_finding": (
                "Month-to-month customers show the "
                "highest churn rate."
            ),
            "possible_business_action": (
                "Target suitable customers with annual "
                "contract incentives."
            ),
            "caution": (
                "Contract type is associated with churn; "
                "causation is not established."
            )
        },
        {
            "analysis_area": "Tenure",
            "key_finding": (
                "Early-tenure customers show higher churn."
            ),
            "possible_business_action": (
                "Improve onboarding and first-month engagement."
            ),
            "caution": (
                "Tenure may interact with contract and services."
            )
        }
    ]
)

eda_findings = pd.concat(
    [eda_findings, new_findings],
    ignore_index=True
)

display(eda_findings)

In [ ]:
eda_output_directory = (
    project_root
    / "reports"
    / "eda_tables"
)

eda_output_directory.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
contract_summary.to_csv(
    eda_output_directory / "churn_by_contract.csv",
    index=False
)

tenure_summary.to_csv(
    eda_output_directory / "churn_by_tenure_group.csv",
    index=False
)

payment_summary.to_csv(
    eda_output_directory / "churn_by_payment_method.csv",
    index=False
)

eda_findings.to_csv(
    eda_output_directory / "eda_findings.csv",
    index=False
)

print("EDA tables saved to:", eda_output_directory)

In [ ]:
monthly_charge_order = [
    "Low",
    "Medium",
    "High",
    "Very High"
]

df["monthly_charge_group"] = pd.Categorical(
    df["monthly_charge_group"],
    categories=monthly_charge_order,
    ordered=True
)

In [ ]:
monthly_charge_summary = (
    df.groupby(
        "monthly_charge_group",
        observed=False
    )
    .agg(
        customer_count=("customerid", "count"),
        churned_customers=("churn_value", "sum"),
        churn_rate=("churn_value", "mean"),
        average_monthly_charge=("monthly_charges", "mean")
    )
    .reset_index()
)

monthly_charge_summary["churn_rate_percentage"] = (
    monthly_charge_summary["churn_rate"] * 100
).round(2)

monthly_charge_summary["average_monthly_charge"] = (
    monthly_charge_summary["average_monthly_charge"]
    .round(2)
)

display(monthly_charge_summary)

In [ ]:
plt.figure(figsize=(9, 5))

bars = plt.bar(
    monthly_charge_summary["monthly_charge_group"].astype(str),
    monthly_charge_summary["churn_rate_percentage"]
)

plt.title("Customer Churn Rate by Monthly Charge Group")
plt.xlabel("Monthly Charge Group")
plt.ylabel("Churn Rate (%)")

for bar, value in zip(
    bars,
    monthly_charge_summary["churn_rate_percentage"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
monthly_charge_by_churn = (
    df.groupby("churn_label")
      .agg(
          customer_count=("customerid", "count"),
          average_monthly_charge=("monthly_charges", "mean"),
          median_monthly_charge=("monthly_charges", "median"),
          minimum_monthly_charge=("monthly_charges", "min"),
          maximum_monthly_charge=("monthly_charges", "max")
      )
      .reset_index()
)

numeric_summary_columns = [
    "average_monthly_charge",
    "median_monthly_charge",
    "minimum_monthly_charge",
    "maximum_monthly_charge"
]

monthly_charge_by_churn[numeric_summary_columns] = (
    monthly_charge_by_churn[numeric_summary_columns]
    .round(2)
)

display(monthly_charge_by_churn)

In [ ]:
not_churned_charges = df.loc[
    df["churn_value"] == 0,
    "monthly_charges"
]

churned_charges = df.loc[
    df["churn_value"] == 1,
    "monthly_charges"
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    [
        not_churned_charges,
        churned_charges
    ],
    tick_labels=[
        "Not Churned",
        "Churned"
    ]
)

plt.title("Monthly Charges by Churn Status")
plt.xlabel("Customer Status")
plt.ylabel("Monthly Charges")

plt.tight_layout()
plt.show()

### Observation

The distribution of monthly charges differs between churned and retained
customers. The median and average charges should be compared to determine
whether churned customers generally pay more.

Monthly charges may be associated with churn, but charges may also be
related to internet-service type, additional services and contract type.
The predictive model will evaluate these variables together.

In [ ]:
internet_service_summary = churn_summary(
    df,
    "internet_service"
)

display(internet_service_summary)

In [ ]:
plot_churn_rate(
    df,
    "internet_service",
    title="Customer Churn Rate by Internet Service"
)

In [ ]:
internet_service_counts = (
    df["internet_service"]
    .value_counts()
    .sort_values(ascending=True)
)

plt.figure(figsize=(9, 5))

bars = plt.barh(
    internet_service_counts.index,
    internet_service_counts.values
)

plt.title("Number of Customers by Internet Service")
plt.xlabel("Number of Customers")
plt.ylabel("Internet Service")

for bar, value in zip(
    bars,
    internet_service_counts.values
):
    plt.text(
        bar.get_width() + 15,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center"
    )

plt.tight_layout()
plt.show()

### Observation

Churn rates vary across internet-service categories. The highest-risk
service group should be investigated further by comparing its contract
type, monthly charges, tenure and technical-support subscriptions.

The result represents an association and should not be interpreted as
proof that the internet-service type directly causes churn.

In [ ]:
online_security_summary = churn_summary(
    df,
    "online_security"
)

display(online_security_summary)

In [ ]:
plot_churn_rate(
    df,
    "online_security",
    title="Customer Churn Rate by Online Security Subscription"
)

In [ ]:
tech_support_summary = churn_summary(
    df,
    "tech_support"
)

display(tech_support_summary)

In [ ]:
plot_churn_rate(
    df,
    "tech_support",
    title="Customer Churn Rate by Technical Support Subscription"
)

### Observation

Customers without online security or technical support may show a higher
churn rate than customers subscribed to these services.

This may indicate that service assistance and customer engagement are
important retention factors. However, these variables may also interact
with internet-service type, contract type and monthly charges.

In [ ]:
conditions = [
    (
        (df["online_security"] == "Yes")
        & (df["tech_support"] == "Yes")
    ),
    (
        (df["online_security"] == "Yes")
        & (df["tech_support"] != "Yes")
    ),
    (
        (df["online_security"] != "Yes")
        & (df["tech_support"] == "Yes")
    ),
    (
        (df["online_security"] == "No")
        & (df["tech_support"] == "No")
    )
]

segments = [
    "Security and Support",
    "Security Only",
    "Support Only",
    "Neither Security nor Support"
]

df["security_support_segment"] = np.select(
    conditions,
    segments,
    default="No Internet Service"
)

In [ ]:
print(
    df["security_support_segment"]
    .value_counts()
)

In [ ]:
security_support_summary = churn_summary(
    df,
    "security_support_segment"
)

display(security_support_summary)

In [ ]:
plot_churn_rate(
    df,
    "security_support_segment",
    title="Churn Rate by Security and Support Subscription"
)

In [ ]:
senior_citizen_summary = churn_summary(
    df,
    "senior_citizen"
)

display(senior_citizen_summary)

In [ ]:
plot_churn_rate(
    df,
    "senior_citizen",
    title="Customer Churn Rate by Senior Citizen Status"
)

In [ ]:
print(df["senior_citizen"].value_counts(dropna=False))

In [ ]:
partner_summary = churn_summary(
    df,
    "partner"
)

display(partner_summary)

In [ ]:
plot_churn_rate(
    df,
    "partner",
    title="Customer Churn Rate by Partner Status"
)

In [ ]:
dependents_summary = churn_summary(
    df,
    "dependents"
)

display(dependents_summary)

In [ ]:
plot_churn_rate(
    df,
    "dependents",
    title="Customer Churn Rate by Dependent Status"
)

### Observation

Churn rates differ across household-related customer groups, including
partner and dependent status.

These characteristics may act as indicators of customer stability, but
they should not automatically be treated as direct causes of churn.
Retention recommendations should focus on controllable factors such as
service quality, contract flexibility, support and pricing.

In [ ]:
gender_summary = churn_summary(
    df,
    "gender"
)

display(gender_summary)

In [ ]:
plot_churn_rate(
    df,
    "gender",
    title="Customer Churn Rate by Gender"
)

In [ ]:
demographic_variables = [
    "gender",
    "senior_citizen",
    "partner",
    "dependents"
]

demographic_summary_tables = []

for column in demographic_variables:
    summary = churn_summary(df, column)
    summary.insert(0, "demographic_variable", column)
    summary = summary.rename(
        columns={
            column: "category"
        }
    )

    demographic_summary_tables.append(summary)

demographic_summary = pd.concat(
    demographic_summary_tables,
    ignore_index=True
)

display(demographic_summary)

In [ ]:
def highest_risk_category(summary, category_column):
    row = summary.sort_values(
        "churn_rate",
        ascending=False
    ).iloc[0]

    return (
        row[category_column],
        row["churn_rate_percentage"]
    )

In [ ]:
highest_charge_group, charge_churn_rate = (
    highest_risk_category(
        monthly_charge_summary,
        "monthly_charge_group"
    )
)

highest_internet_group, internet_churn_rate = (
    highest_risk_category(
        internet_service_summary,
        "internet_service"
    )
)

highest_support_group, support_churn_rate = (
    highest_risk_category(
        tech_support_summary,
        "tech_support"
    )
)

In [ ]:
additional_findings = pd.DataFrame(
    [
        {
            "analysis_area": "Monthly Charges",
            "key_finding": (
                f"{highest_charge_group} monthly-charge customers "
                f"show the highest churn rate at "
                f"{charge_churn_rate:.2f}%."
            ),
            "possible_business_action": (
                "Review pricing, perceived value and suitable "
                "discount options for high-risk customers."
            ),
            "caution": (
                "Monthly charges may be related to service type "
                "and subscribed features."
            )
        },
        {
            "analysis_area": "Internet Service",
            "key_finding": (
                f"{highest_internet_group} customers show the "
                f"highest churn rate at "
                f"{internet_churn_rate:.2f}%."
            ),
            "possible_business_action": (
                "Investigate service experience, pricing and "
                "support needs for this segment."
            ),
            "caution": (
                "Service type is associated with churn, but "
                "causation is not established."
            )
        },
        {
            "analysis_area": "Technical Support",
            "key_finding": (
                f"The '{highest_support_group}' category shows "
                f"the highest churn rate at "
                f"{support_churn_rate:.2f}%."
            ),
            "possible_business_action": (
                "Offer targeted technical-support onboarding "
                "or trial plans to suitable customers."
            ),
            "caution": (
                "Technical-support status may interact with "
                "internet service and customer tenure."
            )
        }
    ]
)

eda_findings = pd.concat(
    [
        eda_findings,
        additional_findings
    ],
    ignore_index=True
)

display(eda_findings)

In [ ]:
monthly_charge_summary.to_csv(
    eda_output_directory
    / "churn_by_monthly_charge_group.csv",
    index=False
)

monthly_charge_by_churn.to_csv(
    eda_output_directory
    / "monthly_charges_by_churn_status.csv",
    index=False
)

internet_service_summary.to_csv(
    eda_output_directory
    / "churn_by_internet_service.csv",
    index=False
)

online_security_summary.to_csv(
    eda_output_directory
    / "churn_by_online_security.csv",
    index=False
)

tech_support_summary.to_csv(
    eda_output_directory
    / "churn_by_tech_support.csv",
    index=False
)

security_support_summary.to_csv(
    eda_output_directory
    / "churn_by_security_support_segment.csv",
    index=False
)

demographic_summary.to_csv(
    eda_output_directory
    / "churn_by_demographics.csv",
    index=False
)

eda_findings.to_csv(
    eda_output_directory
    / "eda_findings.csv",
    index=False
)

print("Additional EDA tables saved to:")
print(eda_output_directory)

In [ ]:
churned_df = df[df["churn_value"] == 1].copy()
retained_df = df[df["churn_value"] == 0].copy()

total_monthly_revenue = df["monthly_charges"].sum()

churned_monthly_revenue = (
    churned_df["monthly_charges"].sum()
)

retained_monthly_revenue = (
    retained_df["monthly_charges"].sum()
)

monthly_revenue_share_churned = (
    churned_monthly_revenue
    / total_monthly_revenue
    * 100
)

revenue_summary = pd.DataFrame(
    {
        "metric": [
            "Total monthly charges",
            "Monthly charges from churned customers",
            "Monthly charges from retained customers",
            "Share represented by churned customers"
        ],
        "value": [
            round(total_monthly_revenue, 2),
            round(churned_monthly_revenue, 2),
            round(retained_monthly_revenue, 2),
            round(monthly_revenue_share_churned, 2)
        ]
    }
)

display(revenue_summary)

In [ ]:
monthly_revenue_by_status = (
    df.groupby("churn_label")
      .agg(
          customer_count=("customerid", "count"),
          total_monthly_charges=("monthly_charges", "sum"),
          average_monthly_charge=("monthly_charges", "mean"),
          median_monthly_charge=("monthly_charges", "median")
      )
      .reset_index()
)

columns_to_round = [
    "total_monthly_charges",
    "average_monthly_charge",
    "median_monthly_charge"
]

monthly_revenue_by_status[columns_to_round] = (
    monthly_revenue_by_status[columns_to_round]
    .round(2)
)

display(monthly_revenue_by_status)

In [ ]:
plt.figure(figsize=(8, 5))

bars = plt.bar(
    monthly_revenue_by_status["churn_label"],
    monthly_revenue_by_status["total_monthly_charges"]
)

plt.title("Total Monthly Charges by Churn Status")
plt.xlabel("Churn Status")
plt.ylabel("Total Monthly Charges")

for bar, value in zip(
    bars,
    monthly_revenue_by_status["total_monthly_charges"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{value:,.0f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
total_charges_summary = (
    df.groupby("churn_label")
      .agg(
          customer_count=("customerid", "count"),
          total_historical_charges=("total_charges", "sum"),
          average_total_charges=("total_charges", "mean"),
          median_total_charges=("total_charges", "median")
      )
      .reset_index()
)

charge_columns = [
    "total_historical_charges",
    "average_total_charges",
    "median_total_charges"
]

total_charges_summary[charge_columns] = (
    total_charges_summary[charge_columns]
    .round(2)
)

display(total_charges_summary)

In [ ]:
plt.figure(figsize=(8, 5))

plt.boxplot(
    [
        retained_df["total_charges"],
        churned_df["total_charges"]
    ],
    tick_labels=[
        "Not Churned",
        "Churned"
    ],
    showfliers=False
)

plt.title("Total Charges by Churn Status")
plt.xlabel("Customer Status")
plt.ylabel("Total Charges")

plt.tight_layout()
plt.show()

In [ ]:
cltv_by_churn = (
    df.groupby("churn_label")
      .agg(
          customer_count=("customerid", "count"),
          total_cltv=("cltv", "sum"),
          average_cltv=("cltv", "mean"),
          median_cltv=("cltv", "median"),
          minimum_cltv=("cltv", "min"),
          maximum_cltv=("cltv", "max")
      )
      .reset_index()
)

cltv_columns = [
    "total_cltv",
    "average_cltv",
    "median_cltv",
    "minimum_cltv",
    "maximum_cltv"
]

cltv_by_churn[cltv_columns] = (
    cltv_by_churn[cltv_columns]
    .round(2)
)

display(cltv_by_churn)

In [ ]:
plt.figure(figsize=(8, 5))

plt.boxplot(
    [
        retained_df["cltv"],
        churned_df["cltv"]
    ],
    tick_labels=[
        "Not Churned",
        "Churned"
    ],
    showfliers=False
)

plt.title("Customer Lifetime Value by Churn Status")
plt.xlabel("Customer Status")
plt.ylabel("CLTV")

plt.tight_layout()
plt.show()

In [ ]:
df["cltv_group"] = pd.qcut(
    df["cltv"],
    q=4,
    labels=[
        "Low CLTV",
        "Medium CLTV",
        "High CLTV",
        "Very High CLTV"
    ],
    duplicates="drop"
)

In [ ]:
print(
    df["cltv_group"]
      .value_counts()
      .sort_index()
)

In [ ]:
cltv_group_summary = (
    df.groupby(
        "cltv_group",
        observed=False
    )
    .agg(
        customer_count=("customerid", "count"),
        churned_customers=("churn_value", "sum"),
        churn_rate=("churn_value", "mean"),
        average_cltv=("cltv", "mean")
    )
    .reset_index()
)

cltv_group_summary["churn_rate_percentage"] = (
    cltv_group_summary["churn_rate"] * 100
).round(2)

cltv_group_summary["average_cltv"] = (
    cltv_group_summary["average_cltv"]
    .round(2)
)

display(cltv_group_summary)

In [ ]:
plt.figure(figsize=(9, 5))

bars = plt.bar(
    cltv_group_summary["cltv_group"].astype(str),
    cltv_group_summary["churn_rate_percentage"]
)

plt.title("Churn Rate by Customer Lifetime Value Group")
plt.xlabel("CLTV Group")
plt.ylabel("Churn Rate (%)")
plt.xticks(rotation=20)

for bar, value in zip(
    bars,
    cltv_group_summary["churn_rate_percentage"]
):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{value:.1f}%",
        ha="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
high_value_threshold = df["cltv"].quantile(0.75)

print(
    "High-value CLTV threshold:",
    round(high_value_threshold, 2)
)

In [ ]:
df["high_value_customer"] = np.where(
    df["cltv"] >= high_value_threshold,
    "High Value",
    "Standard Value"
)

In [ ]:
high_value_summary = (
    df.groupby("high_value_customer")
      .agg(
          customer_count=("customerid", "count"),
          churned_customers=("churn_value", "sum"),
          churn_rate=("churn_value", "mean"),
          total_cltv=("cltv", "sum")
      )
      .reset_index()
)

high_value_summary["churn_rate_percentage"] = (
    high_value_summary["churn_rate"] * 100
).round(2)

high_value_summary["total_cltv"] = (
    high_value_summary["total_cltv"]
    .round(2)
)

display(high_value_summary)

In [ ]:
high_value_churned = df[
    (df["high_value_customer"] == "High Value")
    & (df["churn_value"] == 1)
].copy()

print(
    "High-value customers who churned:",
    len(high_value_churned)
)

print(
    "Combined CLTV of high-value churned customers:",
    round(high_value_churned["cltv"].sum(), 2)
)

In [ ]:
churn_reason_counts = (
    churned_df["churn_reason"]
    .value_counts()
    .rename_axis("churn_reason")
    .reset_index(name="churned_customers")
)

churn_reason_counts["percentage_of_churned"] = (
    churn_reason_counts["churned_customers"]
    / len(churned_df)
    * 100
).round(2)

display(churn_reason_counts)

In [ ]:
top_churn_reasons = (
    churn_reason_counts
    .head(10)
    .sort_values(
        "churned_customers",
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_churn_reasons["churn_reason"],
    top_churn_reasons["churned_customers"]
)

plt.title("Top 10 Recorded Reasons for Customer Churn")
plt.xlabel("Number of Churned Customers")
plt.ylabel("Churn Reason")

for bar, value in zip(
    bars,
    top_churn_reasons["churned_customers"]
):
    plt.text(
        bar.get_width() + 2,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
top_churn_reason_percentages = (
    churn_reason_counts
    .head(10)
    .sort_values(
        "percentage_of_churned",
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_churn_reason_percentages["churn_reason"],
    top_churn_reason_percentages["percentage_of_churned"]
)

plt.title("Top Churn Reasons as a Percentage of Churned Customers")
plt.xlabel("Percentage of Churned Customers (%)")
plt.ylabel("Churn Reason")

for bar, value in zip(
    bars,
    top_churn_reason_percentages["percentage_of_churned"]
):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
reason_contract_summary = (
    churned_df.groupby(
        ["contract", "churn_reason"]
    )
    .size()
    .reset_index(name="churned_customers")
)

display(
    reason_contract_summary
    .sort_values(
        "churned_customers",
        ascending=False
    )
    .head(20)
)

In [ ]:
reason_contract_pivot = (
    reason_contract_summary
    .pivot(
        index="churn_reason",
        columns="contract",
        values="churned_customers"
    )
    .fillna(0)
)

display(
    reason_contract_pivot
    .assign(
        total=lambda data: data.sum(axis=1)
    )
    .sort_values(
        "total",
        ascending=False
    )
    .head(10)
)

In [ ]:
city_summary = (
    df.groupby("city")
      .agg(
          customer_count=("customerid", "count"),
          churned_customers=("churn_value", "sum"),
          churn_rate=("churn_value", "mean"),
          total_monthly_charges=("monthly_charges", "sum"),
          total_cltv=("cltv", "sum")
      )
      .reset_index()
)

city_summary["churn_rate_percentage"] = (
    city_summary["churn_rate"] * 100
).round(2)

city_summary[
    [
        "total_monthly_charges",
        "total_cltv"
    ]
] = city_summary[
    [
        "total_monthly_charges",
        "total_cltv"
    ]
].round(2)

display(
    city_summary
    .sort_values(
        "customer_count",
        ascending=False
    )
    .head(10)
)

In [ ]:
top_cities_by_churn_count = (
    city_summary
    .sort_values(
        "churned_customers",
        ascending=False
    )
    .head(10)
    .sort_values(
        "churned_customers",
        ascending=True
    )
)

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_cities_by_churn_count["city"],
    top_cities_by_churn_count["churned_customers"]
)

plt.title("Top Cities by Number of Churned Customers")
plt.xlabel("Number of Churned Customers")
plt.ylabel("City")

for bar, value in zip(
    bars,
    top_cities_by_churn_count["churned_customers"]
):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        str(value),
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
minimum_city_customers = 30

eligible_cities = city_summary[
    city_summary["customer_count"]
    >= minimum_city_customers
].copy()

In [ ]:
top_cities_by_churn_rate = (
    eligible_cities
    .sort_values(
        "churn_rate_percentage",
        ascending=False
    )
    .head(10)
    .sort_values(
        "churn_rate_percentage",
        ascending=True
    )
)

display(
    top_cities_by_churn_rate[
        [
            "city",
            "customer_count",
            "churned_customers",
            "churn_rate_percentage"
        ]
    ]
)

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_cities_by_churn_rate["city"],
    top_cities_by_churn_rate[
        "churn_rate_percentage"
    ]
)

plt.title(
    f"Highest City Churn Rates "
    f"(Minimum {minimum_city_customers} Customers)"
)

plt.xlabel("Churn Rate (%)")
plt.ylabel("City")

for bar, value in zip(
    bars,
    top_cities_by_churn_rate[
        "churn_rate_percentage"
    ]
):
    plt.text(
        bar.get_width() + 0.2,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.1f}%",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
zip_code_summary = (
    df.groupby(
        [
            "zip_code",
            "city",
            "latitude",
            "longitude"
        ],
        dropna=False
    )
    .agg(
        customer_count=("customerid", "count"),
        churned_customers=("churn_value", "sum"),
        churn_rate=("churn_value", "mean"),
        total_monthly_charges=("monthly_charges", "sum"),
        total_cltv=("cltv", "sum")
    )
    .reset_index()
)

zip_code_summary["churn_rate_percentage"] = (
    zip_code_summary["churn_rate"] * 100
).round(2)

zip_code_summary[
    [
        "total_monthly_charges",
        "total_cltv"
    ]
] = zip_code_summary[
    [
        "total_monthly_charges",
        "total_cltv"
    ]
].round(2)

display(
    zip_code_summary
    .sort_values(
        "customer_count",
        ascending=False
    )
    .head()
)

In [ ]:
correlation_columns = [
    "senior_citizen",
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "cltv",
    "churn_value"
]

available_correlation_columns = [
    column
    for column in correlation_columns
    if column in df.columns
    and pd.api.types.is_numeric_dtype(df[column])
]

correlation_matrix = (
    df[available_correlation_columns]
    .corr()
    .round(2)
)

display(correlation_matrix)

In [ ]:
plt.figure(figsize=(8, 6))

image = plt.imshow(
    correlation_matrix,
    aspect="auto"
)

plt.colorbar(image)

plt.xticks(
    range(len(correlation_matrix.columns)),
    correlation_matrix.columns,
    rotation=45,
    ha="right"
)

plt.yticks(
    range(len(correlation_matrix.index)),
    correlation_matrix.index
)

for row_index in range(len(correlation_matrix.index)):
    for column_index in range(
        len(correlation_matrix.columns)
    ):
        plt.text(
            column_index,
            row_index,
            correlation_matrix.iloc[
                row_index,
                column_index
            ],
            ha="center",
            va="center"
        )

plt.title("Correlation Matrix of Numerical Variables")
plt.tight_layout()
plt.show()

In [ ]:
contract_tenure_churn = (
    df.groupby(
        [
            "contract",
            "tenure_group"
        ],
        observed=False
    )
    ["churn_value"]
    .mean()
    .mul(100)
    .reset_index(
        name="churn_rate_percentage"
    )
)

display(contract_tenure_churn)

In [ ]:
contract_tenure_pivot = (
    contract_tenure_churn
    .pivot(
        index="contract",
        columns="tenure_group",
        values="churn_rate_percentage"
    )
)

display(
    contract_tenure_pivot.round(2)
)

In [ ]:
plt.figure(figsize=(10, 4))

image = plt.imshow(
    contract_tenure_pivot,
    aspect="auto"
)

plt.colorbar(
    image,
    label="Churn Rate (%)"
)

plt.xticks(
    range(len(contract_tenure_pivot.columns)),
    contract_tenure_pivot.columns.astype(str),
    rotation=30,
    ha="right"
)

plt.yticks(
    range(len(contract_tenure_pivot.index)),
    contract_tenure_pivot.index
)

for row_index in range(
    len(contract_tenure_pivot.index)
):
    for column_index in range(
        len(contract_tenure_pivot.columns)
    ):
        value = contract_tenure_pivot.iloc[
            row_index,
            column_index
        ]

        if pd.notna(value):
            plt.text(
                column_index,
                row_index,
                f"{value:.1f}%",
                ha="center",
                va="center"
            )

plt.title("Churn Rate by Contract and Tenure Group")
plt.xlabel("Tenure Group")
plt.ylabel("Contract Type")

plt.tight_layout()
plt.show()

In [ ]:
top_reason_row = churn_reason_counts.iloc[0]

top_reason = top_reason_row["churn_reason"]

top_reason_percentage = (
    top_reason_row["percentage_of_churned"]
)

highest_city_row = (
    eligible_cities
    .sort_values(
        "churn_rate_percentage",
        ascending=False
    )
    .iloc[0]
)

In [ ]:
final_findings = pd.DataFrame(
    [
        {
            "analysis_area": "Historical Revenue",
            "key_finding": (
                f"Customers who churned represented "
                f"{monthly_revenue_share_churned:.2f}% "
                f"of the observed monthly charges."
            ),
            "possible_business_action": (
                "Prioritize retention segments based on "
                "both churn risk and customer value."
            ),
            "caution": (
                "This is historical churn analysis, not "
                "future predicted revenue at risk."
            )
        },
        {
            "analysis_area": "Churn Reason",
            "key_finding": (
                f"The most frequently recorded churn reason "
                f"was '{top_reason}', accounting for "
                f"{top_reason_percentage:.2f}% of churned "
                f"customers."
            ),
            "possible_business_action": (
                "Design retention actions that directly "
                "address frequently reported reasons."
            ),
            "caution": (
                "Churn reason is recorded after churn and "
                "must not be used as a prediction feature."
            )
        },
        {
            "analysis_area": "Geography",
            "key_finding": (
                f"Among cities with at least "
                f"{minimum_city_customers} customers, "
                f"{highest_city_row['city']} showed the "
                f"highest churn rate at "
                f"{highest_city_row['churn_rate_percentage']:.2f}%."
            ),
            "possible_business_action": (
                "Investigate service quality, pricing and "
                "competition in high-churn locations."
            ),
            "caution": (
                "Geographic comparisons require sufficient "
                "sample size and do not prove causation."
            )
        }
    ]
)

eda_findings = pd.concat(
    [
        eda_findings,
        final_findings
    ],
    ignore_index=True
)

display(eda_findings)

In [ ]:
revenue_summary.to_csv(
    eda_output_directory
    / "historical_revenue_summary.csv",
    index=False
)

monthly_revenue_by_status.to_csv(
    eda_output_directory
    / "monthly_revenue_by_churn_status.csv",
    index=False
)

total_charges_summary.to_csv(
    eda_output_directory
    / "total_charges_by_churn_status.csv",
    index=False
)

cltv_by_churn.to_csv(
    eda_output_directory
    / "cltv_by_churn_status.csv",
    index=False
)

cltv_group_summary.to_csv(
    eda_output_directory
    / "churn_by_cltv_group.csv",
    index=False
)

high_value_summary.to_csv(
    eda_output_directory
    / "high_value_customer_summary.csv",
    index=False
)

churn_reason_counts.to_csv(
    eda_output_directory
    / "churn_reason_summary.csv",
    index=False
)

reason_contract_summary.to_csv(
    eda_output_directory
    / "churn_reason_by_contract.csv",
    index=False
)

city_summary.to_csv(
    eda_output_directory
    / "city_churn_summary.csv",
    index=False
)

zip_code_summary.to_csv(
    eda_output_directory
    / "zip_code_churn_summary.csv",
    index=False
)

correlation_matrix.to_csv(
    eda_output_directory
    / "numerical_correlation_matrix.csv"
)

contract_tenure_churn.to_csv(
    eda_output_directory
    / "churn_by_contract_and_tenure.csv",
    index=False
)

eda_findings.to_csv(
    eda_output_directory
    / "eda_findings.csv",
    index=False
)

print("All EDA tables saved to:")
print(eda_output_directory)

In [ ]:
enriched_analytics_path = (
    project_root
    / "data"
    / "processed"
    / "telco_churn_analytics_enriched.csv"
)

enriched_analytics_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(
    enriched_analytics_path,
    index=False
)

print("Enriched analytics dataset saved to:")
print(enriched_analytics_path)

In [ ]:
eda_completion_summary = pd.DataFrame(
    {
        "analysis": [
            "Overall churn",
            "Contract",
            "Tenure",
            "Payment method",
            "Monthly charges",
            "Internet service",
            "Online security",
            "Technical support",
            "Security-support segment",
            "Demographics",
            "Historical revenue",
            "Total charges",
            "CLTV",
            "Churn reasons",
            "Geography",
            "Contract-tenure interaction"
        ],
        "completed": [True] * 16
    }
)

display(eda_completion_summary)